# Continuous Wave Analysis of NANOGrav 15-year dataset

This notebook reproduces (some) results from the NANOGrav 15-year CW analysis.

### Requirements

- This notebook should be executed on a (NVIDIA) GPU with CUDA-enabled JAX.

- The analysis requires the packages imported in the following cell, and other common packages.

- By default, single precision (`float32`) is used. If desired, use double precision by modifying the `__init__.py` file, but this slows down the analysis.

In [ ]:
# to load/save objects
import pickle
import h5py

# for plotting
import numpy as np
import matplotlib.pyplot as plt
import corner

# for sampling
import jax
import jax.random as jr
import numpyro

# prometheus objects
from prometheus.spectral_models import IndependentSpectralModel, CommonSpectralModel
from prometheus import spectra
from prometheus.deterministic_models import DeterministicModel
from prometheus import deterministic
from prometheus.pta_model import PTAModel

%load_ext autoreload
%autoreload 2

In [ ]:
# check that we are running on a GPU
# this should print something like "CudaDevice(id=...)"
print(jax.devices())

## Load Data

Prometheus stores PTA data in a custom `prometheus.data.Data` object. This object may be constructed from a list of `enterprise.Pulsar.PintPulsar` objects and a white noise dictionary.

The `Data` object needs only be constructed once per dataset, after which it can be stored and used repeatedly under different models. The only exception is when we wish to change the number of frequency bins used in the pulsar noise or deterministic signal model. In which case, the `Data` object needs to be built from scratch with a different number of frequency bins.

In [ ]:
# load previously created data object (see gwb_pe.ipynb)
with open('../data/NG15/data.pkl', 'rb') as fp:
    NG15_data = pickle.load(fp)

## Construct Spectral Models

$\texttt{Prometheus}$ requires two spectral models to perform parameter estimation:

1) `prometheus.spectral_models.IndependentSpectralModel`: 
- This corresponds to the intrinsic pulsar noise model.
- It assumes the same spectral model (e.g. power law) is applied independently to every pulsar in the array.

2) `prometheus.spectral_models.CommonSpectralModel`:
- This corresponds to a gravitational wave background, or some common stochastic process.
- A common spectrum is applied to all pulsars under some correlation matrix.
- HD and CURN correlations are supported, or you can plug in your own pulsar correlation matrix.

Each spectral model corresponds to a Gaussian process for the Fourier coefficients which represent the stochastic timing residuals. The spectral models require a callable input: `get_phi_diag_func`. This function should take an array of spectral parameters and an array frequencies as input, and outputs the diagonal elements of the covariance matrix used in the prior on the Fourier coefficients. Common `get_phi_diag_func` are available in `prometheus.spectra.py`, or you can build your own custom spectrum!

Advanced users need not adhere to the two-spectral-model-requirement above. Instead they can create their own custom `prometheus.spectral_models.SpectralModel`. This takes a bit more work - see the advanced modeling example notebooks.

In [ ]:
# we'll model the pulsar noise with a power law
# the IndependentSpectralModel automatically applies this model to
# every pulsar in the array independently
psr_model = IndependentSpectralModel(name='irn_pl',
                                     get_phi_diag_func=spectra.power_law,
                                     parameter_bounds=[[-20., -12.], # log_amp bounds
                                                       [0., 7.]],    # spectral index bounds
                                     data=NG15_data)

# model the GWB with a power law (no inter-pulsar correlations)
# also specify the number of frequency bins used in the GWB model
gwb_model = CommonSpectralModel(name='gwb_pl',
                                    get_phi_diag_func=spectra.power_law,
                                    parameter_bounds=[[-20., -12.], # log_amp bounds
                                                        [0., 7.]],    # spectral index bounds
                                    correlation_matrix='CURN',
                                    data=NG15_data,
                                    nfreqs=14)

## Specify deterministic model

$\texttt{Prometheus}$ supports the construction of arbitrary deterministic models, whose parameters will be jointly inferred with those of spectral models. To add a deterministic signal, we must build a `prometheus.deterministic_models.DeterministicModel` object. The most important attribute of this object is a `get_delays_func` which returns the timing delays induced by the deterministic signal (across puslars) given a set of deterministic model parameters. The `get_delays_func` for an evolving continuous wave from an individual circular binary (with pulsar terms) can be found in `prometheus.deterministic.py`.

Users can construct a custom `get_delays_func` to make their own deterministic models. However, they should verify their model is stable in single precision and accurately represented in a Fourier basis (see the `tests` folder).

We will use the evolving CW model to compare $\texttt{Prometheus}$ and [QuickCW](https://github.com/nanograv/QuickCW) on the NANOGrav 15-year dataset. The CW model parameters are grouped 'cw_source', 'psr_phases', and 'psr_dists'. The 'cw_source' parameters consist of (in order): $\log_{10}(\mathcal{M}\,\,[M_\odot]), \;\log_{10}(f_{CW}\,\,[\text{Hz}]), \;\cos{\iota},\; \psi, \;\log_{10} h,\; \cos{\theta}, \;\phi, \;\Phi_0$. i.e. chirp mass, initial frequency, inclination, polarization, amplitude, polar sky location, azimuthal sky location, and initial phase, respectively. The 'psr_phases' and 'psr_dists' parameters are the phase of the CW at each pulsar, and the pulsar distances respectively.

In [ ]:
# the minimum/maximum values of the source parameters described above
# the pulsar distance and phase parameters are sampled automatically
cw_source_mins = np.array([7., -8.7, -1., 0, -18., -1., 0., 0.])
cw_source_maxs = np.array([9., -8.2, 1., np.pi, -12., 1., 2. * np.pi, 2. * np.pi])
cw_parameter_bounds = np.vstack((cw_source_mins, cw_source_maxs)).T

# now we can build the deterministic model 
cw_model = DeterministicModel(name='cw_evolve',
                              data=NG15_data,
                              get_delays_func=deterministic.cw_delay_evolve_float32,
                              parameter_bounds=cw_parameter_bounds,
                              with_psr_params=True)

## Build a PTA model

Now we put everything together in a `prometheus.pta_model.PTAModel` object.

In [ ]:
pta_model = PTAModel(psr_model=psr_model,
                     gwb_model=gwb_model,
                     det_model=cw_model)
print(type(pta_model))

In [ ]:
# we can get a dictionary of parameter names
# and required shapes for each parameter

# in this case there are 30 frequency bins and 67 pulsars,
# so we sample (67, 2 x 30) Fourier coefficients,
# (67, 2) pulsar power law parameters,
# 2 GWB power law parameters,
# 8 CW source parameters,
# 67 pulsar phase and distance parameters.

print(pta_model.get_param_names_and_shapes())

In [ ]:
# let's test the posterior evaluation
# first we'll draw a random set of parameter values
param_dict = pta_model.draw_params_from_prior()

# and we can evaluate the posterior at these test parameters
print(pta_model.ln_posterior(param_dict['xi'],
                             param_dict['irn_pl'],
                             param_dict['gwb_pl'],
                             param_dict['cw_evolve'],
                             param_dict['psr_phases'],
                             param_dict['psr_dists']))

In [ ]:
# let's time the posterior
# on a NVIDIA GeForce RTX 3090, the evaluation is ~400 micro-seconds
%timeit pta_model.ln_posterior(param_dict['xi'], param_dict['irn_pl'], param_dict['gwb_pl'], param_dict['cw_evolve'], param_dict['psr_phases'], param_dict['psr_dists'])

## Sample the posterior!

One method of the `prometheus.pta_model.PTAModel` object is a `NumPyro` probabilistic sampling model. We'll use `NumPyro`'s No U-Turn Sampler (NUTS), an extension of Hamiltonian Monte Carlo (HMC) to sample the posterior. For an introduction to HMC sampling, see e.g. [this paper](https://arxiv.org/abs/1701.02434).

On a decent GPU, the cell below should take ~30 minutes to sample.

Here we use a CURN GWB model to compare our results with [QuickCW](https://github.com/nanograv/QuickCW). However, $\texttt{Prometheus}$ is able to jointly model a CW and HD-correlated background (simply change the correlation pattern in the `CommonSpectralModel` above).

__WARNING__: On occasion, the NUTS sampler will warm-up in a non-representative region of parameter space, which causes poor sampling. You can identify this failure while the sampler runs by examining the step-size. If the step-size falls below $\sim 10^{-5}$, terminate the run and try again - this time with a different random seed. You should find an average step-size of $\sim 10^{-2}$ for this dataset and model.

In [ ]:
# get probabilistic sampling model from the PTA model
sampling_model = pta_model.sampling_model

# build the NumPyro NUTS kernel
nuts_kernel = numpyro.infer.NUTS(model=sampling_model,
                                 target_accept_prob=0.9)

# specify MCMC attributes
mcmc = numpyro.infer.MCMC(sampler=nuts_kernel,
                          num_warmup=1000,
                          num_samples=5000)

# seed to start sampling
# (see "WARNING" above: change if necessary)
seed = 557

# run MCMC and get samples
mcmc.run(jr.PRNGKey(seed))
samples = mcmc.get_samples()

In [ ]:
# uncomment to examine sampling diagnostics
# (this is a long output because we sample 
# ALL the Fourier coefficients)

# mcmc.print_summary()

In [ ]:
# # save samples to feather file
# utils.save_chain(samples_dict=samples,
#                  filepath='chains/NG15_CURN_PL_Nf14_CW.feather')

# # we can also load samples
# # samples = utils.load_chain('chains/NG15_CURN_PL_Nf14_CW.feather')

## Post-processing

Let's examine the samples and make some plots. The samples are stored in a Python dictionary. The parameter names (dictionary keys) are the names of the spectral and deterministic models specified above.

The CW model parameters are 'cw_evolve', 'psr_phases', and 'psr_dists'. The 'cw_evolve' parameters consist of (in order): $\log_{10}(\mathcal{M}\,\,[M_\odot]), \;\log_{10}(f_{CW}\,\,[\text{Hz}]), \;\cos{\iota},\; \psi, \;\log_{10} h,\; \cos{\theta}, \;\phi, \;\Phi_0$. i.e. chirp mass, initial frequency, inclination, polarization, amplitude, polar sky location, azimuthal sky location, and initial phase, respectively. The 'psr_phases' and 'psr_dists' parameters are the phase of the CW at each pulsar, and the pulsar distances respectively.

There will also be samples of parameters 'a' and 'xi'. These are the Fourier coefficients and the 'standardized' Fourier coefficients (which use a standard normal prior), respectively. As opposed to other PTA analysis software, like $\texttt{Enterprise}$ which analytically marginalizes the Fourier coefficients, $\texttt{Prometheus}$ numerically marginalizes over the Fourier coefficients by sampling them. This means the total number of parameters is much larger than usual (but not too large for HMC)!

In [ ]:
print(samples.keys())
print(f'shape of GWB samples = {samples["gwb_pl"].shape}')  # (num samples, num params)
print(f'shape of IRN samples = {samples["irn_pl"].shape}')  # (num samples, num psrs, num params)
print(f'shape of Fourier coefficient samples = {samples["a"].shape}')  # (num samples, num psrs, 2 * num freqs)
print(f'shape of CW source parameter samples = {samples["cw_evolve"].shape}')  # (num samples, 8)
print(f'shape of pulsar phase parameter samples = {samples["psr_phases"].shape}')  # (num samples, num psrs,)
print(f'shape of pulsar distance samples = {samples["psr_dists"].shape}')  # (num samples, num psrs,)

In [ ]:
# we're going to compare our results with quickCW,
# so we'll load the quickCW samples from NG15 here
with h5py.File('chains/15yr_quickCW_detection.h5', 'r') as f:
    quickCW_samples_all = f['samples_cold'][:]
    quickCW_samples = quickCW_samples_all[0, :, :8]
    # restrict frequency and chirp mass range to match our samples
    mask = np.where(np.logical_and(np.logical_and(quickCW_samples[:, 3] < -8.2,
                                                  quickCW_samples[:, 3] > -8.7),
                                   quickCW_samples[:, 5] < 9.))[0]
    quickCW_samples = quickCW_samples[mask]

In [ ]:
# trace plot of CW frequency parameter
plt.figure(figsize=(10, 5))
plt.plot(samples['cw_evolve'][:, 1])
plt.xlabel('HMC iteration')
plt.ylabel('parameter value')
plt.title(r'Trace plot of $\log_{10} f_\text{CW}\;\;[\text{Hz}]$')
plt.show()

In [ ]:
# plot CW recovery under CURN GWB model

# match parameter ordering of quickCW
ordering = np.array([-3, 2, -2, 1, 4, 0])

cw_source_labels = np.array([r'$\log_{10}(\mathcal{M}\,\,[M_\odot])$',
                             r'$\log_{10}(f_{CW}\,\,[\text{Hz}])$',
                             r'$\cos{\iota}$', r'$\psi$', r'$\log_{10} h$',
                             r'$\cos{\theta}$', r'$\phi$', r'$\Phi_0$'])

# plot samples from Prometheus
cw_samples = np.array(samples['cw_evolve'])
fig = corner.corner(cw_samples[:, ordering],
                    bins=30,
                    labels=cw_source_labels[ordering],
                    label_kwargs={'fontsize':16},
                    color='C0',
                    hist_kwargs={'density': True})

# plot samples from QuickCW
corner.corner(data=quickCW_samples[:, :-2],
              bins=30,
              fig=fig,
              color='C1',
              hist_kwargs={'density': True})

# plot legend/title
axes = np.array(fig.axes).reshape((len(ordering), len(ordering)))
axes[0, -1].plot([], [], color='C0', label='Prometheus samples')
axes[0, -1].plot([], [], color='C1', label='QuickCW samples')
axes[0, -1].legend(fontsize=20, loc='lower right')
fig.suptitle('NG15 CW recovery (GWB modeled as CURN)',
             fontsize=24)
fig.show()

## Bonus:

By default, $\texttt{Prometheus}$ uses normal priors on the pulsar distance parameters, but those familiar with CW searches know that more technical priors are used (Arzoumanian et al. 2023 Eq. 20 & 21). We can modify the pulsar distance priors. First we'll code up our own prior density function. Then this prior enters the `PTAModel` as `add_ln_factor` which gets added to the log-posterior evaluation.

In [ ]:
import jax.numpy as jnp

# which pulsar distances are measured with parallax and DM
where_PX = jnp.where(NG15_data.psr_dist_method == 'PX')[0]
where_DM = jnp.where(NG15_data.psr_dist_method == 'DM')[0]

# measured pulsar distance and uncertainty
psr_dists_measured = jnp.array(NG15_data.psr_dists_measured)
psr_dists_std = jnp.array(NG15_data.psr_dists_std)

# the additional prior weighting needs to accept all the parameters of the joint model,
# even if it only uses a subset of the model parameters
def psr_dists_lnprior(pn_params, gwb_params, cw_source_params, psr_phases, psr_dists):

    # prior on pulsar distance when measured with parallax
    lnpriors_PX_dist = jax.vmap(lambda x, y, z : deterministic.ln_p_PX(x, y, z),
                                in_axes=(0, 0, 0))(psr_dists, psr_dists_measured, psr_dists_std)
    lnprior_PX_summed = jnp.sum(lnpriors_PX_dist[where_PX])

    # prior on pulsar distance when measured with DM
    lnpriors_DM_dist = jax.vmap(lambda x, y, z : deterministic.ln_p_DM(x, y, z),
                                in_axes=(0, 0, 0))(psr_dists, psr_dists_measured, psr_dists_std)
    lnprior_DM_summed = jnp.sum(lnpriors_DM_dist[where_DM])

    # Prometheus still samples under a normal prior, so sneakily subtract that density here
    ln_normal_correction = 0.5 * jnp.sum((psr_dists - psr_dists_measured)**2 / psr_dists_std**2)

    # new "advanced" pulsar distance prior
    return lnprior_PX_summed + lnprior_DM_summed + ln_normal_correction


# new PTA model with modified pulsar distance prior
pta_model_with_psr_dist_prior = PTAModel(psr_model=psr_model,
                                         gwb_model=gwb_model,
                                         det_model=cw_model,
                                         add_ln_factor=psr_dists_lnprior)